# Multiple diffraction: what changes with path order?
Replay the **17 October 2026, 60 GHz** study: two wood columns and three leaf
surfaces, four links, V/H polarization, maximum total interaction depth four.
Reflection and transmission were enabled; diffuse scattering was disabled.

DiffeRT 0.12.0 and Sionna RT 2.2.0 have different edge eligibility and field models.
The primary quantity here is **incoherent excess loss**. Coherent results are retained
in the source table but remain exploratory because duplicate complex paths introduce
additional ambiguity. No absolute vegetation-loss accuracy is established.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/replay.py").is_file())
sys.path.insert(0, str(ROOT / "notebooks"))
from replay import verify_data, read_json, dataframe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.2, "figure.dpi": 110})
manifest = verify_data()
print(f"Verified {len(manifest['files'])} bundled historical data files.")


In [ ]:
results = dataframe("multiple_diffraction_60ghz/results_table.csv")
analysis = read_json("multiple_diffraction_60ghz/analysis.json")
verification = read_json("multiple_diffraction_60ghz/verification.json")
assert verification["status"] == "passed"
differt = results.loc[results.solver == "DiffeRT"].copy()
losses = differt.pivot(index=["case", "polarization"], columns="max_diffractions",
                       values="excess_incoherent_db")
assert len(losses) == 8 and list(losses.columns) == [0, 1, 2, 3, 4]
assert losses.notna().all().all()
display(losses.rename(columns=lambda n: f"up to {n} diffractions"))


In [ ]:
CASE = "mixed_left"  # mixed_centre, mixed_right, blocked_coupled
assert CASE in losses.index.get_level_values("case")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for pol, color in [("V", "#137c8b"), ("H", "#d18738")]:
    row = losses.loc[(CASE, pol)]
    axes[0].plot(row.index, row, "o-", color=color, label=pol)
    axes[1].plot([2, 3, 4], row[[2, 3, 4]], "o-", color=color, label=pol)
for axis in axes:
    axis.set(xlabel="Maximum diffraction order", ylabel="Incoherent excess loss (dB)")
    axis.legend(title="Polarization")
axes[0].set_title(f"{CASE}: all orders (total depth ≤ 4)")
axes[0].set_xticks([0, 1, 2, 3, 4])
axes[1].set_title("Zoom after the second diffraction")
axes[1].set_xticks([2, 3, 4])
plt.show()


In [ ]:
increments = pd.DataFrame({
    "loss reduction, order 1 to 2 (dB)": losses[1] - losses[2],
    "loss reduction, order 2 to 4 (dB)": losses[2] - losses[4],
    "loss reduction, order 3 to 4 (dB)": losses[3] - losses[4],
})
display(increments)
assert (losses[2] - losses[4]).abs().max() < 0.16
assert (losses[3] - losses[4]).abs().max() < 0.001
print("Largest absolute order-2-to-4 change:", (losses[2] - losses[4]).abs().max(), "dB")


## Audit the search evidence
A small order increment is meaningful only within the tested search controls.
The saved checks compare exhaustive/hybrid enumeration through depth three,
4-million versus 16-million DiffeRT visibility rays, and Sionna ray counts/seeds.
Full exhaustive depth-four attempts were stopped; those are not claimed complete.


In [ ]:
checks = pd.DataFrame(analysis["checks"])
exhaustive = checks.loc[checks["check"] == "hybrid_vs_exhaustive_depth3"]
visibility = checks.loc[checks["check"] == "differt_visibility_4m_vs_16m"]
high_ray = checks.loc[(checks["check"] == "sionna_by_ray_count") & (checks["rays"] == 64000000)]
assert len(exhaustive) == len(visibility) == len(high_ray) == 8
assert (exhaustive[["missing_in_hybrid", "extra_in_hybrid"]] == 0).all().all()
assert visibility["power_delta_db"].abs().max() < 1e-8
assert high_ray["loss_range_db"].max() < 2e-5
display(high_ray[["case", "polarization", "rays", "seeds", "loss_range_db"]])
print("Input hashes retained for", len(analysis["input_inventory"]), "paired links.")


## The native totals have different path families
The table below retains full Sionna totals and its wood-edge-only diagnostic beside
DiffeRT. Sionna's open-leaf-edge contributions are absent from this DiffeRT version.
Consequently, differences do not isolate diffraction order or establish which solver
is physically more accurate. The [next notebook](03_fullwave_validation.ipynb) adds
independent but explicitly reduced full-wave controls.


In [ ]:
final = results.loc[(results.solver != "DiffeRT") | (results.max_diffractions == 4)]
display(final.pivot(index=["case", "polarization"], columns="solver",
                    values="excess_incoherent_db"))
display(differt.groupby("max_diffractions")[["paths", "duplicate_power_bound_db"]].max())
print(verification["unvalidated"])
